# 04 — Backpropagation

**CS231n Lecture 4.** The most important notebook in the series. Everything
after it is layers stacked on this one idea.

You derived gradients by hand in notebook 02 and it worked, because a linear
classifier is one operation. Try that for a 50-layer network and you will fail.
Backprop replaces "differentiate the whole expression" with a **local** rule
applied repeatedly:

> Every node knows how to turn the gradient arriving at its output into the
> gradient at its inputs, using only the values it saw on the way forward.
> Nothing else.

| You write | The idea |
|---|---|
| `add_backward`, `mul_backward`, `max_backward` | the three gate patterns |
| `sigmoid_backward`, `relu_backward` | a smooth gate and a gating gate |
| `matmul_backward` | the shape trick — the one you'll use forever |
| `staged_forward` / `staged_backward` | a full circuit, by hand |
| `branching_backward` | the rule people get wrong: reused values **add** |

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from cs231n_utils import (rel_error, grade, show_solution, use,
                          eval_numerical_gradient_array, setup_plots, SERIES,
                          INK, INK_2, INK_MUTED, GRID, SURFACE)

setup_plots()
np.random.seed(0)

## 1. The chain rule, as a local rule

Take $f(x,y,z) = (x+y)z$ at $x=-2, y=5, z=-4$. Introduce a name for the
intermediate, $q = x+y$, so $f = qz$.

**Forward:** $q = 3$, $f = -12$.

**Backward**, starting from $\frac{\partial f}{\partial f} = 1$:

$$ \frac{\partial f}{\partial q} = z = -4 \qquad \frac{\partial f}{\partial z} = q = 3 $$

$$ \frac{\partial f}{\partial x} = \frac{\partial f}{\partial q}\frac{\partial q}{\partial x} = -4 \cdot 1 = -4 \qquad \frac{\partial f}{\partial y} = -4 \cdot 1 = -4 $$

The pattern, and it is the whole algorithm:

$$ \underbrace{\frac{\partial L}{\partial \text{input}}}_{\text{what you pass back}} = \underbrace{\frac{\partial L}{\partial \text{output}}}_{\text{upstream, handed to you}} \times \underbrace{\frac{\partial \text{output}}{\partial \text{input}}}_{\text{local, you know this}} $$

The `+` node never needs to know it feeds a `*` node, or that a softmax sits
1000 layers downstream. It gets a number, multiplies by its local derivative,
passes it on. **That locality is what makes deep networks trainable.**

In [ ]:
# The example, concretely.
x, y, z = -2.0, 5.0, -4.0
q = x + y
f = q * z
print("forward:  q = %.0f   f = %.0f" % (q, f))

df = 1.0
dq, dz = df * z, df * q        # f = q*z   -> multiply swaps
dx, dy = dq * 1.0, dq * 1.0    # q = x+y   -> add distributes
print("backward: dx = %.0f  dy = %.0f  dz = %.0f" % (dx, dy, dz))

### The same example, as the graph it actually is

Grey values above each edge are the forward pass; orange values below each
edge are what gets carried backward along it. `q`'s node sends `dq = -4`
backward along **both** of its incoming edges (to `x` and to `y`) unchanged —
watch for that pattern, it is the "add" gate from the table below.

In [ ]:
fig, ax = plt.subplots(figsize=(6.6, 3.0))
nodes = {'x': (0.3, 2.2), 'y': (0.3, 0.8), 'z': (0.3, -1.0),
        'q': (3.0, 1.5), 'f': (5.7, 0.6)}
node_labels = {'x': 'x = -2', 'y': 'y = 5', 'z': 'z = -4', 'q': 'q = x+y', 'f': 'f = q·z'}
gates = {'q', 'f'}

for name, (px, py) in nodes.items():
    is_gate = name in gates
    ax.add_patch(plt.Circle((px, py), 0.44 if is_gate else 0.26,
                            facecolor=SURFACE, edgecolor=INK if is_gate else GRID,
                            linewidth=1.6 if is_gate else 1.2, zorder=3))
    ax.annotate(node_labels[name], xy=(px, py), ha='center', va='center',
               fontsize=8, weight='semibold' if is_gate else 'normal', zorder=4)

edges = [('x', 'q', 'x = -2', 'dx = -4'), ('y', 'q', 'y = 5', 'dy = -4'),
        ('q', 'f', 'q = 3', 'dq = -4'), ('z', 'f', 'z = -4', 'dz = 3')]
for src, dst, fwd, bwd in edges:
    x0, y0 = nodes[src]; x1, y1 = nodes[dst]
    ax.annotate("", xy=(x1, y1), xytext=(x0, y0),
               arrowprops=dict(arrowstyle='-|>', color=INK_MUTED, lw=1.3,
                               shrinkA=17, shrinkB=21))
    mx, my = (x0 + x1) / 2, (y0 + y1) / 2
    ax.annotate(fwd, xy=(mx, my + 0.22), ha='center', fontsize=7.5, color=INK_2)
    ax.annotate(bwd, xy=(mx, my - 0.28), ha='center', fontsize=7.5, color=SERIES[1],
               weight='semibold')

ax.annotate("df = 1  (start here)", xy=(6.5, 0.6), fontsize=7.5, color=SERIES[1],
           weight='semibold', va='center')
ax.set_xlim(-0.2, 7.6); ax.set_ylim(-1.7, 2.9)
ax.set_aspect('equal'); ax.axis('off')
ax.set_title("Forward (grey, above each edge) / backward (orange, below)")
fig.tight_layout()
plt.show()

## 2. The three gate patterns

Memorise these. You will recognise them in every architecture you read.

| Gate | Forward | Backward | Nickname |
|---|---|---|---|
| **add** | $z = x + y$ | both inputs get `dout`, unchanged | **distributor** |
| **mul** | $z = xy$ | each input gets `dout` times *the other input* | **swapper** |
| **max** | $z = \max(x,y)$ | all of `dout` to the larger input, `0` to the other | **router** |

Two consequences worth holding on to:

- The **multiply** gate means that if one input is tiny, the *other* input's
  gradient is tiny. This is exactly why unnormalised inputs wreck training: a
  pixel at 255 and a pixel at 0.001 produce wildly different weight gradients
  for no good reason.
- The **max** gate is why ReLU networks are sparse in the backward pass. A unit
  that was off contributes nothing and learns nothing this step.

### TODO 1 — the three gates

Each takes the upstream gradient `dout` plus the forward inputs, and returns the
gradient with respect to each input. Assume elementwise numpy arrays. For `max`,
send ties to `x`.

In [ ]:
def add_backward(dout, x, y):
    """z = x + y  ->  (dx, dy)"""
    pass  # TODO

def mul_backward(dout, x, y):
    """z = x * y  ->  (dx, dy)"""
    pass  # TODO

def max_backward(dout, x, y):
    """z = maximum(x, y)  ->  (dx, dy).  Ties go to x."""
    pass  # TODO

In [ ]:
for nm, fn in [('add_backward', add_backward), ('mul_backward', mul_backward),
               ('max_backward', max_backward)]:
    grade('nb04', nm, fn, verbose=False)

### Check them the way you'd check real code

`eval_numerical_gradient_array` is the array-valued sibling of the checker from
notebook 03: given a forward function and an upstream gradient, it produces the
backward pass numerically. This is how you validate **any** layer you ever
write.

In [ ]:
ab = use('nb04', 'add_backward', add_backward)
mb = use('nb04', 'mul_backward', mul_backward)
xb = use('nb04', 'max_backward', max_backward)

rng = np.random.RandomState(0)
X, Y, dout = rng.randn(4, 3), rng.randn(4, 3), rng.randn(4, 3)

for name, fwd, bwd in [('add', lambda a, b: a + b, ab),
                       ('mul', lambda a, b: a * b, mb),
                       ('max', lambda a, b: np.maximum(a, b), xb)]:
    dx, dy = bwd(dout, X, Y)
    nx = eval_numerical_gradient_array(lambda t: fwd(t, Y), X.copy(), dout)
    ny = eval_numerical_gradient_array(lambda t: fwd(X, t), Y.copy(), dout)
    print("%-4s  dx rel error %.2e    dy rel error %.2e"
          % (name, rel_error(dx, nx), rel_error(dy, ny)))

## 3. Sigmoid — and why you express the derivative via the output

$$ \sigma(x) = \frac{1}{1+e^{-x}} \qquad \frac{d\sigma}{dx} = \sigma(x)\,(1 - \sigma(x)) $$

You could decompose the sigmoid into `exp`, `+1`, `1/x` and backprop through
four nodes. Or you can treat it as one node with that clean local gradient. Both
give the same answer; the second is faster and numerically better behaved.

**The practical point:** the derivative is written in terms of the *output*
$\sigma(x)$, not the input $x$. Since the forward pass already computed
$\sigma(x)$, you cache and reuse it. Every framework's sigmoid does this.

### TODO 2 — sigmoid and ReLU backward

In [ ]:
def sigmoid_backward(dout, x):
    """s = 1/(1+exp(-x))  ->  dx"""
    pass  # TODO

def relu_backward(dout, x):
    """r = max(0, x)  ->  dx.  What happens at exactly x = 0?"""
    pass  # TODO

In [ ]:
grade('nb04', 'sigmoid_backward', sigmoid_backward, verbose=False)
grade('nb04', 'relu_backward', relu_backward, verbose=False)

In [ ]:
sb = use('nb04', 'sigmoid_backward', sigmoid_backward)
rb = use('nb04', 'relu_backward', relu_backward)

xs = rng.randn(5, 4)
d = rng.randn(5, 4)
print("sigmoid dx rel error %.2e" % rel_error(
    sb(d, xs), eval_numerical_gradient_array(lambda t: 1 / (1 + np.exp(-t)), xs.copy(), d)))
print("relu    dx rel error %.2e" % rel_error(
    rb(d, xs), eval_numerical_gradient_array(lambda t: np.maximum(0, t), xs.copy(), d)))

In [ ]:
# Why sigmoid saturates -- the thing that killed deep nets before ReLU.
xr = np.linspace(-10, 10, 400)
s = 1 / (1 + np.exp(-xr))
fig, ax = plt.subplots(figsize=(6.2, 3.4))
ax.plot(xr, s, color=SERIES[0], label='sigmoid(x)')
ax.plot(xr, s * (1 - s), color=SERIES[1], label="local gradient s(1-s)")
ax.axhline(0.25, color='#8a8887', linewidth=0.9, linestyle='--')
ax.annotate("max local gradient = 0.25", xy=(-9.5, 0.27), fontsize=8, color='#52514e')
ax.set_title("Sigmoid and its local gradient")
ax.set_xlabel("x"); ax.set_ylabel("value")
ax.legend(); ax.set_axisbelow(True)
plt.show()

print("local gradient at x=6:  %.6f" % (1/(1+np.exp(-6)) * (1 - 1/(1+np.exp(-6)))))
print("after 10 stacked layers: %.3e" % ((1/(1+np.exp(-6)) * (1 - 1/(1+np.exp(-6)))) ** 10))

**The local gradient never exceeds 0.25**, and out in the tails it is
effectively zero. Stack ten sigmoids and the best case is $0.25^{10} \approx
10^{-6}$; the realistic case is far worse. The gradient arriving at the early
layers is annihilated.

That is the **vanishing gradient problem**, and it is why networks stayed
shallow for two decades. ReLU's local gradient is exactly `1` wherever the unit
is on — stack a hundred and the gradient is still `1`. Notebook 07 goes further
into this.

## 4. Vectorised backprop and the shape trick

For $Y = XW$ with `X (N,D)` and `W (D,M)`, the true Jacobian
$\partial Y / \partial X$ has shape `(N, M, N, D)`. For a modest layer that is
terabytes. **You never build it.** The answer collapses to two matrix
multiplies:

$$ \frac{\partial L}{\partial X} = \frac{\partial L}{\partial Y} W^\top \qquad\qquad \frac{\partial L}{\partial W} = X^\top \frac{\partial L}{\partial Y} $$

### The shape trick

You do not have to remember which is which. Derive it from shapes every time:

- `dX` **must** have the shape of `X`, which is `(N, D)`
- you have `dout (N, M)` and `W (D, M)`
- the only way to make `(N, D)` out of those is `dout @ W.T` — `(N,M)@(M,D)`

Do the same for `dW`: it must be `(D, M)`, and `X.T @ dout` is `(D,N)@(N,M)`.
There is exactly one arrangement that type-checks, and it is always the right
one.

> This is not a hack, it's a consequence: gradients live in the same space as
> the thing they differentiate, so shapes constrain the answer to uniqueness.
> Use it. It is faster and more reliable than re-deriving indices.

### TODO 3 — `matmul_backward`

In [ ]:
def matmul_backward(dout, x, w):
    """out = x.dot(w).   x (N,D), w (D,M), dout (N,M)  ->  (dx, dw)

    Use the shape trick. Two lines.
    """
    pass  # TODO

In [ ]:
grade('nb04', 'matmul_backward', matmul_backward, verbose=False)

In [ ]:
mm = use('nb04', 'matmul_backward', matmul_backward)
Xm, Wm = rng.randn(6, 4), rng.randn(4, 3)
dm = rng.randn(6, 3)
dx, dw = mm(dm, Xm, Wm)
print("dx shape %s (x is %s)   rel error %.2e"
      % (dx.shape, Xm.shape,
         rel_error(dx, eval_numerical_gradient_array(lambda t: t.dot(Wm), Xm.copy(), dm))))
print("dw shape %s (w is %s)   rel error %.2e"
      % (dw.shape, Wm.shape,
         rel_error(dw, eval_numerical_gradient_array(lambda t: Xm.dot(t), Wm.copy(), dm))))

### The Jacobian you never build — literally, this once

`matmul_backward` works from two matrix multiplies, and the shape trick tells
you *which* multiplies, but not *why* that collapse is even possible. Here is
why, at a toy size where the full Jacobian is only `6x6` instead of
terabytes. Build it the honest way — perturb each entry of `X`, watch how the
whole flattened `Y = X.dot(W)` moves — and look at what falls out.

In [ ]:
Nj, Dj, Mj = 3, 2, 2
Xj = rng.randn(Nj, Dj)
Wj = rng.randn(Dj, Mj)

J = np.zeros((Nj * Mj, Nj * Dj))          # rows: flattened Y, cols: flattened X
h = 1e-5
for a in range(Nj * Dj):
    xp, xm = Xj.ravel().copy(), Xj.ravel().copy()
    xp[a] += h; xm[a] -= h
    yp = (xp.reshape(Nj, Dj).dot(Wj)).ravel()
    ym = (xm.reshape(Nj, Dj).dot(Wj)).ravel()
    J[:, a] = (yp - ym) / (2 * h)

fig, ax = plt.subplots(figsize=(4.3, 4.3))
lim = np.abs(J).max()
im = ax.imshow(J, cmap='RdBu_r', vmin=-lim, vmax=lim)
for k in range(1, Nj):
    ax.axhline(k * Mj - 0.5, color=INK, linewidth=1.0)
    ax.axvline(k * Dj - 0.5, color=INK, linewidth=1.0)
ax.set_title(r"$\partial Y/\partial X$ for $Y=XW$ -- %dx%d, mostly zero" % J.shape)
ax.set_xlabel("flattened X   (N\u00d7D = %d)" % (Nj * Dj))
ax.set_ylabel("flattened Y   (N\u00d7M = %d)" % (Nj * Mj))
ax.set_xticks(range(Nj * Dj)); ax.set_yticks(range(Nj * Mj))
ax.grid(False)
fig.colorbar(im, ax=ax, shrink=0.8)
fig.tight_layout()
plt.show()

print("W.T:\n", np.round(Wj.T, 3))
print("\ntop-left %dx%d block of the Jacobian (should match W.T):" % (Mj, Dj))
print(np.round(J[:Mj, :Dj], 3))

**Read the picture, not just the numbers.** The Jacobian is almost entirely
zero, and what survives is `N` identical copies of `W.T` sitting on the block
diagonal — outlined above. That structure is forced by the model: `Y[n, :]`
depends only on `X[n, :]`, never on any other row, so example `n`'s block of
outputs can only have nonzero derivatives with respect to example `n`'s block
of inputs. Every off-diagonal block is a whole `M x D` slab of exact zeros.

`dX = dout.dot(W.T)` is precisely the operation "multiply each row of `dout`
by the same `W.T` block" — which is exactly what this matrix looks like, just
executed without ever materialising the zeros. That is the whole savings: not
an approximation, a refusal to store what you already know is zero.

## 5. Staged computation — a whole circuit by hand

$$ f(w, x) = \frac{1}{1 + e^{-(w_0 x_0 + w_1 x_1 + w_2)}} $$

This is one sigmoid neuron. The discipline that makes it easy:

1. On the forward pass, **give every intermediate a name** and keep it.
2. On the backward pass, walk the names in reverse, applying one local rule each.

Never try to write $\partial f / \partial w_0$ as a single expression. Staging
is not a crutch — it is the algorithm.

### TODO 4 — forward and backward

In [ ]:
def staged_forward(w0, w1, w2, x0, x1):
    """Return (f, cache) where cache holds every intermediate you'll need.

    Use exactly this staging so the cache matches the reference:
        a = w0 * x0
        b = w1 * x1
        c = a + b
        d = c + w2
        f = sigmoid(d)
    and return  (f, (w0, w1, w2, x0, x1, a, b, c, d, f))
    """
    pass  # TODO

def staged_backward(cache):
    """Walk the cache backwards. Return (dw0, dw1, dw2, dx0, dx1).

    Start from df = 1.0 and apply, in order:
      sigmoid -> add -> add -> multiply, multiply
    """
    w0, w1, w2, x0, x1, a, b, c, d, f = cache
    pass  # TODO

In [ ]:
grade('nb04', 'staged_forward', staged_forward, verbose=False)
grade('nb04', 'staged_backward', staged_backward, verbose=False)

In [ ]:
sf = use('nb04', 'staged_forward', staged_forward)
sbk = use('nb04', 'staged_backward', staged_backward)

vals = np.array([2.0, -3.0, -3.0, -1.0, -2.0])   # w0 w1 w2 x0 x1
out, cache = sf(*vals)
grads = np.array(sbk(cache))
print("f = %.6f" % out)
print("analytic  dw0 %.5f  dw1 %.5f  dw2 %.5f  dx0 %.5f  dx1 %.5f" % tuple(grads))

# numeric check over all five scalar inputs
num = np.zeros(5)
h = 1e-6
for i in range(5):
    vp, vm = vals.copy(), vals.copy()
    vp[i] += h; vm[i] -= h
    num[i] = (sf(*vp)[0] - sf(*vm)[0]) / (2 * h)
print("numeric   dw0 %.5f  dw1 %.5f  dw2 %.5f  dx0 %.5f  dx1 %.5f" % tuple(num))
print("\nrel error %.2e" % rel_error(grads, num))

## 6. The rule people actually get wrong

**When a value feeds more than one downstream node, its gradients ADD.**

This follows from the multivariable chain rule: if $y$ influences $L$ through
two paths,

$$ \frac{\partial L}{\partial y} = \frac{\partial L}{\partial u}\frac{\partial u}{\partial y} + \frac{\partial L}{\partial v}\frac{\partial v}{\partial y} $$

In code this means `dy += ...` and **never** `dy = ...` for the second
contribution. Writing `=` silently drops a term — no crash, no shape error, just
a wrong gradient and a model that trains slightly badly forever.

This is precisely why your `svm_loss_naive` in notebook 02 accumulated
`dW[:, y[i]] -= X[i]` inside the class loop instead of assigning once: the
correct-class weight vector participates in every margin term in the row.

### TODO 5 — `branching_backward`

$$ f(x, y, z) = (x + y) \cdot \max(y, z) $$

`y` appears twice. Get both contributions.

In [ ]:
def branching_backward(dout, x, y, z):
    """f = (x + y) * max(y, z)  ->  (dx, dy, dz).

    y flows through BOTH the add and the max. Ties in max go to y.
    """
    pass  # TODO

In [ ]:
grade('nb04', 'branching_backward', branching_backward, verbose=False)

In [ ]:
bb = use('nb04', 'branching_backward', branching_backward)

xa, ya, za = rng.randn(4, 3), rng.randn(4, 3), rng.randn(4, 3)
da = rng.randn(4, 3)
dxa, dya, dza = bb(da, xa, ya, za)
fb = lambda X, Y, Z: (X + Y) * np.maximum(Y, Z)
print("dx rel error %.2e" % rel_error(dxa, eval_numerical_gradient_array(lambda t: fb(t, ya, za), xa.copy(), da)))
print("dy rel error %.2e" % rel_error(dya, eval_numerical_gradient_array(lambda t: fb(xa, t, za), ya.copy(), da)))
print("dz rel error %.2e" % rel_error(dza, eval_numerical_gradient_array(lambda t: fb(xa, ya, t), za.copy(), da)))

In [ ]:
# What the bug looks like when you use = instead of +=
def branching_backward_BUGGY(dout, x, y, z):
    s, m = x + y, np.maximum(y, z)
    ds, dm = dout * m, dout * s
    dx, dy = ds, ds
    y_wins = (y >= z)
    dy = dm * y_wins          # BUG: overwrites instead of adding
    return dx, dy, dm * (~y_wins)

_, dy_bug, _ = branching_backward_BUGGY(da, xa, ya, za)
dy_true = eval_numerical_gradient_array(lambda t: fb(xa, t, za), ya.copy(), da)
print("buggy dy rel error: %.3f   <- big, but nothing crashed" % rel_error(dy_bug, dy_true))
print("correct dy rel error: %.2e" % rel_error(dya, dy_true))

Note what the bug did: **no exception, no shape mismatch, correct-looking
output.** The only way to catch this class of error is a numeric gradient check.
Run one on every layer you write, the day you write it.

## 7. Recap

1. State the local rule in one sentence.
2. What are the add, multiply and max gates nicknamed, and why?
3. Why does an unnormalised input hurt, in terms of the multiply gate?
4. Sigmoid's local gradient peaks at what value? What does that imply for depth?
5. Why is the sigmoid derivative written in terms of its output?
6. You have `dout (N,M)`, `X (N,D)`, `W (D,M)`. Derive `dW` from shapes alone.
7. Why must you never materialise the Jacobian of a matmul?
8. A value feeds three downstream nodes. What do you do with the three
   incoming gradients, and what is the symptom if you get it wrong?

In [ ]:
# show_solution('nb04', 'matmul_backward')
# show_solution('nb04', 'staged_backward')
# show_solution('nb04', 'branching_backward')